In [ ]:
import os
from pathlib import Path
from math import sqrt, floor, ceil

import numpy as np
import pandas as pd

from skimage.io import imread
from skimage.transform import rotate
from skimage import morphology, measure
from skimage.segmentation import find_boundaries


# ============================================================
# 1. Project paths
# ============================================================

def find_project_root():
    """
    Finds the project root by looking for:
    data/imgs
    data/masks/masks

    This makes the script work on GitHub or on another computer,
    as long as the folder structure is preserved.
    """

    try:
        start = Path(__file__).resolve().parent
    except NameError:
        start = Path.cwd().resolve()

    possible_starts = [start, Path.cwd().resolve()]

    for start_path in possible_starts:
        for parent in [start_path] + list(start_path.parents):
            img_dir = parent / "data" / "imgs"
            mask_dir = parent / "data" / "masks" / "masks"

            if img_dir.exists() and mask_dir.exists():
                return parent

    raise FileNotFoundError(
        "Could not find the project root. Make sure the repo contains "
        "'data/imgs' and 'data/masks/masks'."
    )


PROJECT_ROOT = find_project_root()

IMG_DIR = PROJECT_ROOT / "data" / "imgs"
MASK_DIR = PROJECT_ROOT / "data" / "masks" / "masks"
OUTPUT_DIR = PROJECT_ROOT / "data" / "Separate csv of features"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_CSV = OUTPUT_DIR / "features_all.csv"


# ============================================================
# 2. Valid image extensions
# ============================================================

VALID_EXTENSIONS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}


# ============================================================
# 3. Mask loading and preprocessing
# ============================================================

def load_mask(mask_path):
    """
    Reads a mask image and converts it into a binary mask.
    White pixels become True.
    Black pixels become False.
    """

    mask = imread(mask_path)

    if mask.ndim == 3:
        mask = mask[:, :, 0]

    if mask.max() > 1:
        mask = mask / 255.0

    mask_bin = mask > 0.5

    return mask_bin


def preprocess_mask(mask_bin):
    """
    Cleans the binary mask by removing small noise and filling small holes.
    """

    mask_bin = morphology.remove_small_objects(mask_bin, min_size=100)
    mask_bin = morphology.remove_small_holes(mask_bin, area_threshold=100)

    return mask_bin


def get_main_lesion(mask_bin):
    """
    Keeps only the largest connected lesion region.
    This avoids small noisy regions affecting the features.
    """

    labels = measure.label(mask_bin)
    regions = measure.regionprops(labels)

    if len(regions) == 0:
        return None, None

    largest = max(regions, key=lambda r: r.area)
    lesion = labels == largest.label

    return lesion, largest


# ============================================================
# 4. Border and shape features
# ============================================================

def extract_border_shape_features(mask_bin):
    """
    Extracts lesion features based on the binary mask.
    """

    cleaned_mask = preprocess_mask(mask_bin)
    lesion, region = get_main_lesion(cleaned_mask)

    if lesion is None:
        return None

    border = find_boundaries(lesion, mode="inner")

    area = region.area
    perimeter = region.perimeter
    total_pixels = lesion.shape[0] * lesion.shape[1]

    features = {
        "total_pixels": total_pixels,

        # Lesion area in pixels
        "area": area,

        # Percentage of the image occupied by the lesion
        "lesion_percentage": area / total_pixels if total_pixels > 0 else np.nan,

        # Length of the lesion border
        "perimeter": perimeter,

        # Compactness = 1 means close to a perfect circle.
        # Higher values indicate more irregular or elongated shapes.
        "compactness": (perimeter ** 2) / (4 * np.pi * area) if area > 0 else np.nan,

        # Number of pixels on the lesion boundary
        "border_pixels": int(np.sum(border))
    }

    return features


# ============================================================
# 5. Asymmetry functions
# ============================================================

def MidPoint(mask):
    row_summed = np.sum(mask, axis=1)
    col_summed = np.sum(mask, axis=0)

    half_row = np.sum(row_summed) / 2
    half_col = np.sum(col_summed) / 2

    row_mid = next(
        i for i, n in enumerate(np.add.accumulate(row_summed))
        if n > half_row
    )

    col_mid = next(
        i for i, n in enumerate(np.add.accumulate(col_summed))
        if n > half_col
    )

    return row_mid, col_mid


def cut_mask(mask):
    col_sums = np.sum(mask, axis=0)
    row_sums = np.sum(mask, axis=1)

    active_cols = [i for i, v in enumerate(col_sums) if v != 0]
    active_rows = [i for i, v in enumerate(row_sums) if v != 0]

    if not active_rows or not active_cols:
        return None

    return mask[
        active_rows[0]:active_rows[-1] + 1,
        active_cols[0]:active_cols[-1] + 1
    ]


def asymmetry(mask):
    row_mid, col_mid = MidPoint(mask)

    upper = mask[:ceil(row_mid), :]
    lower = mask[floor(row_mid):, :]

    left = mask[:, :ceil(col_mid)]
    right = mask[:, floor(col_mid):]

    flipped_lower = np.flip(lower, axis=0)
    flipped_right = np.flip(right, axis=1)

    min_r = min(upper.shape[0], flipped_lower.shape[0])
    min_c = min(left.shape[1], flipped_right.shape[1])

    hxor = np.logical_xor(
        upper[:min_r, :],
        flipped_lower[:min_r, :]
    )

    vxor = np.logical_xor(
        left[:, :min_c],
        flipped_right[:, :min_c]
    )

    total = np.sum(mask)

    if total == 0:
        return None

    score = (np.sum(hxor) + np.sum(vxor)) / (total * 2)

    return round(float(score), 4)


def feature_asymmetry(mask, n=8):
    """
    Rotates the mask several times between 0 and 90 degrees.
    Computes asymmetry for each rotation.
    Returns the average asymmetry score.
    """

    scores = []

    for i in range(n):
        degrees = 90 * i / n

        rotated = rotate(
            mask.astype(np.float64),
            degrees,
            cval=0.0
        )

        binarized = (rotated > 0.5).astype(np.uint8)
        cut = cut_mask(binarized)

        if cut is None:
            continue

        score = asymmetry(cut)

        if score is not None and 0 <= score <= 1:
            scores.append(score)

    if not scores:
        return None, 0

    return round(float(np.mean(scores)), 4), len(scores)


# ============================================================
# 6. Match image to mask
# ============================================================

def find_mask_for_image(image_path):
    """
    Finds the corresponding mask for an image.

    Example:
    image: PAT_1000_31_620.png
    mask:  PAT_1000_31_620_mask.png
    """

    image_stem = image_path.stem

    possible_mask_names = []

    for ext in VALID_EXTENSIONS:
        possible_mask_names.append(f"{image_stem}_mask{ext}")
        possible_mask_names.append(f"{image_stem}{ext}")

    for mask_name in possible_mask_names:
        candidate = MASK_DIR / mask_name

        if candidate.exists():
            return candidate

    return None


# ============================================================
# 7. Main processing loop
# ============================================================

def process_all_images(n_rotations=8):
    rows = []

    image_files = [
        file for file in sorted(IMG_DIR.iterdir())
        if file.suffix.lower() in VALID_EXTENSIONS
    ]

    print(f"Images found: {len(image_files)}")

    for image_path in image_files:
        mask_path = find_mask_for_image(image_path)

        if mask_path is None:
            print(f"Skipped {image_path.name}: no matching mask found")
            continue

        try:
            mask_bin = load_mask(mask_path)

            if np.sum(mask_bin) < 50:
                print(f"Skipped {image_path.name}: mask has fewer than 50 white pixels")
                continue

            shape_features = extract_border_shape_features(mask_bin)

            if shape_features is None:
                print(f"Skipped {image_path.name}: no valid lesion found")
                continue

            asymmetry_score, rotations_used = feature_asymmetry(
                mask_bin,
                n=n_rotations
            )

            row = {
                "image_name": image_path.name,
                "mask_name": mask_path.name,
                **shape_features,
                "asymmetry_score": asymmetry_score,
                "rotations_used": rotations_used
            }

            rows.append(row)

        except Exception as e:
            print(f"Error processing {image_path.name}: {e}")

    df = pd.DataFrame(rows)

    df = df.sort_values("image_name").reset_index(drop=True)

    df.to_csv(OUTPUT_CSV, index=False)

    print(f"\nSaved {len(df)} rows to:")
    print(OUTPUT_CSV)

    return df


# ============================================================
# 8. Entry point
# ============================================================

if __name__ == "__main__":
    features_df = process_all_images(n_rotations=8)

    print("\nPreview:")
    print(features_df.head())

Images found: 2298
Skipped PAT_100_393_595.png: no matching mask found
Skipped PAT_1014_85_22.png: no matching mask found
Skipped PAT_1019_110_194.png: no matching mask found
Skipped PAT_1026_124_206.png: no matching mask found
Skipped PAT_1031_148_29.png: no matching mask found
Skipped PAT_1031_148_292.png: no matching mask found
Skipped PAT_1034_160_819.png: no matching mask found
Skipped PAT_1046_203_511.png: no matching mask found
Skipped PAT_104_1755_320.png: mask has fewer than 50 white pixels
Skipped PAT_1063_268_704.png: no matching mask found
Skipped PAT_1063_269_755.png: no matching mask found
Skipped PAT_1065_275_136.png: no matching mask found
Skipped PAT_1065_277_768.png: no matching mask found
Skipped PAT_1071_313_789.png: no matching mask found
Skipped PAT_1072_318_856.png: no matching mask found
Skipped PAT_1093_379_273.png: no matching mask found
Skipped PAT_1094_381_355.png: no matching mask found
Skipped PAT_1112_455_845.png: no matching mask found
Skipped PAT_1119_4